In [2]:
# Load model directly
from transformers import AutoTokenizer, AutoModelForCausalLM

tokenizer = AutoTokenizer.from_pretrained("nvidia/Nemotron-Research-Reasoning-Qwen-1.5B")
model = AutoModelForCausalLM.from_pretrained("nvidia/Nemotron-Research-Reasoning-Qwen-1.5B")
#messages = [
    #{"role": "user", "content": "Who are you?"},
#]
#inputs = tokenizer.apply_chat_template(
	#messages,
	#add_generation_prompt=True,
	#tokenize=True,
	#return_dict=True,
	#return_tensors="pt",
#).to(model.device)

#outputs = model.generate(**inputs, max_new_tokens=40)
#print(tokenizer.decode(outputs[0][inputs["input_ids"].shape[-1]:]))

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/485 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/729 [00:00<?, ?B/s]

2025-12-05 21:32:45.836638: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-12-05 21:32:45.902684: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-12-05 21:32:47.142682: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


...

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/2.11G [00:00<?, ?B/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/5.00G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

In [6]:
import analogy_dataset as ad
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch
import pandas as pd
from tqdm import tqdm

In [8]:
def build_prompt(row):
    return f"""You are a multilingual translator.
Translate this analogy MCQ entirely from Hindi to Tamil.

Question: {row['Part1']} : {row['Part2']} :: {row['Part3']} : ?
Options:
A. {row['Option_1']}
B. {row['Option_2']}
C. {row['Option_3']}
D. {row['Option_4']}
"""

# ---------------------------------------------------------
# Step 3: Translation Function (Mistral chat mode)
# ---------------------------------------------------------
def get_translation(prompt):
    messages = [
        {"role": "user", "content": prompt}
    ]

    encoded = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        return_tensors="pt",
        enable_thinking = False
    ).to(model.device)

    output_ids = model.generate(
        encoded,
        max_new_tokens=512,
        do_sample=False
    )

    # remove prompt tokens → keep only generated part
    generated_text = tokenizer.decode(
        output_ids[0][encoded.shape[-1]:],
        skip_special_tokens=True
    )

    return generated_text.strip()

# ---------------------------------------------------------
# Step 4: Load Analogy Dataset
# ---------------------------------------------------------
all_rows = []
iterator = ad.get_iterator()

while True:
    row = iterator.get_next_row()
    if not row:
        break
    all_rows.append(row)

batch_size = 20
total = len(all_rows)

# choose which batch to run:
batch_index = 0

start = batch_index * batch_size
end = min(start + batch_size, total)
batch_rows = all_rows[start:end]

translated_data = []

for row in tqdm(batch_rows):
    prompt = build_prompt(row)

    try:
        tamil = get_translation(prompt)
    except Exception as e:
        tamil = f"ERROR: {e}"
        print("Error at:", row["Formatted_Question"], e)

    translated_data.append({
        "Formatted_Question": row["Formatted_Question"],
        "Answer_Hindi": row["All_Correct_Answers"],
        "Tamil_Translated": tamil
    })

# ---------------------------------------------------------
# Step 5: Save Translated Batch
# ---------------------------------------------------------
df = pd.DataFrame(translated_data)
df.to_csv(f"tats_batch_{batch_index}_mistral7b.csv", index=False)

print(f"Batch {batch_index} saved successfully!")

100%|███████████████████████████████████████████████████████████████████████████████████| 20/20 [15:28<00:00, 46.41s/it]

Batch 0 saved successfully!
